Projet : Anticipez les besoins en consommation de bâtiments
Notebook : Exploration des données - feature engineering
Created by: Thomas Durand-Texte, Feb. 2023

# Import des packages et données
## import des packages

In [ ]:
import os

import pandas as pd
from pandas import IndexSlice as idx

import pickle

import numpy as np
# import dask as dd
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
import datetime as dt
import scipy.stats as st

import missingno as msno

import pingouin as pg
from sklearn import linear_model
from sklearn import model_selection, metrics, preprocessing


import matplotlib.pyplot as plt
import seaborn as sns

cm = 1./2.54

## Paramètres graphiques et fonctions utiles

In [ ]:
import subprocess

white_font = True
def set_theme( white_font=True ):
    """ set_theme( white_font=True ) """
    if white_font: wht, grey, blck = '0.84' , '0.5', 'k'
    else: wht, grey, blck = 'k', '0.5', '0.84'
    rc = { 'figure.facecolor':(0.118,)*3,
            'axes.labelcolor':wht,
            'axes.edgecolor':wht,
            'axes.facecolor':(0,0,0,0),
            'text.color':'white',
            'text.usetex':False,
            'text.latex.preamble':r'\usepackage[cm]{sfmath} \usepackage{amsmath}' ,
            'font.family': 'sans-serif' ,
            'font.sans-serif': 'DejaVu Sans' ,
            'xtick.color':wht,
            'ytick.color':wht,
            "axes.grid" : True,
            "grid.color": (0.7,)*3,
            "grid.linewidth": 0.4,
            "grid.linestyle": (10,5),
            'legend.edgecolor':'0.2',
            'legend.facecolor':(0.2,0.2,0.2,0.6),
            # 'legend.framealpha':'0.6',
            'pdf.fonttype':42,
            'savefig.format':'pdf',
            'savefig.transparent':True,
            'figure.dpi':150, # for better agreemet figsize vs real size
        }

    base_palette = sns.color_palette()
    sns.set_theme( 'notebook' , rc=rc )
    sns.set_palette( base_palette )
    return


def make_folder( path_folder ):
    path_folder = path_folder.__str__()
    try:
        if os.path.isdir( path_folder ) : return
        os.makedirs(path_folder)
    except OSError:
        pass
    return

def concat_folders(*args, **kwargs):
    """ concat_folders(*args, **kwargs)
        concatenate folders in args (strings) """
    sPath = ''
    for arg in args:
        if arg == '..': sPath = sPath[:sPath[:-1].rfind(os.sep)+1]
        else: sPath += arg
        if sPath[-1] != os.sep: sPath += os.sep
    return sPath

class Path(object):
    """ Path( s_in='', s_lim=None)
        create a path to the string s_in (default is current path)
        and stops after s_lim """
    n_Path = 0
    def __init__(self, s_in='', s_lim=None):
        """docstring."""
        if s_in == '': s_in = os.getcwd()
        if not s_lim is None:
            if s_lim in s_in:
                s_in = s_in[ :s_in.index( s_lim ) + len(s_lim) ]
        self.sPath = concat_folders(s_in)
        self.N = Path.n_Path
        Path.n_Path += 1

    def __add__(self, other):
        """ Path + str : return str """
        if isinstance(other, str): return self.sPath + other

    def __truediv__(self, other):
        """ Path / str : return path concatenated"""
        if isinstance(other, str): return Path(concat_folders(self.sPath, other))

    def __invert__(self):
        """ ~Path : return str of the path """
        return self.sPath

    def __str__(self):
        """ __str__ return str of the path """
        return self.sPath
    # __str__ #

    def makedir( self ):
        return make_folder( self )


def gs_opt( filename ):
    """ otpimisation of a pdf file with gosthscript """
    filenameTmp = filename.replace('.pdf', '') + '_tmp.pdf'
    gs = ['gs',
            '-sDEVICE=pdfwrite',
            '-dEmbedAllFonts=true',
            '-dSubsetFonts=true',             # Create font subsets (default)
            '-dPDFSETTINGS=/prepress',        # Image resolution
            '-dDetectDuplicateImages=true',   # Embeds images used multiple times only once
            '-dCompressFonts=true',           # Compress fonts in the output (default)
            '-dNOPAUSE',                      # No pause after each image
            '-dQUIET',                        # Suppress output
            '-dBATCH',                        # Automatically exit
            '-sOutputFile='+filenameTmp,      # Save to temporary output
            filename]                         # Input file

    subprocess.run(gs)                                      # Create temporary file
    subprocess.run( 'rm -f ' + filename, shell=True)            # Delete input file
    subprocess.run( 'mv -f ' + filenameTmp + " " + filename, shell=True) # Rename temporary to input file

def savefig( fig, savename, **kwargs ):
    """ savefig( fig, savename, **kwargs )
        Saves a figure with kwargs (fig.savefig( savename, **kwargs) ).
        A check is done first to determine if a folder has to be created according to savename.
        Finally, if the file is saved as .pdf, gosthscript optimisation is performed. """
    if os.sep in savename: make_folder( savename[:savename.rindex(os.sep)] )
    fig.savefig( savename, **kwargs )
    savename += '.pdf'
    if os.path.isfile( savename ): gs_opt( savename )


def image_size_from_width_and_shape( width: float, shape: tuple, ymargin=0. ):
    """ return tuple (width, height) corresponding to image shape """
    return width, width*shape[0]/shape[1]+ymargin

def image_size_from_height_and_shape( height: float, shape: tuple, xmargin=0. ):
    """ return tuple (width, height) corresponding to image shape """
    return height*shape[1]/shape[0]+xmargin, height


set_theme()
del set_theme

## Chargement des données

Affichage de l'arborescence

In [ ]:
def print_listdir( path=None, level=0, exclude=['ressources']) :
    suffix = ''
    if level > 0:
        suffix = ' |-'* level
    vals = os.listdir( path )
    vals.sort()
    if path is None:
        path = ''
    for val in vals:
        if val in exclude: continue
        print( suffix, val)
        if os.path.isdir( path + val ):
            print_listdir( path + val + '/', level+1, exclude )

print_listdir( exclude=['.venv', 'ressources','models', 'devel'] )

1. Chargement des données
2. lower strings
3. compression et sauvegarde des données

In [ ]:
path = 'data/source/'
filename = '2016_Building_Energy_Benchmarking'
compression = 'gzip'

if True:
    df = pd.read_csv( path + filename + '.csv' )
    for key in df.keys():
        if df[key].dtype == 'object':
            df[key] = df[key].str.lower()

    # suppression des colonnes vides (ici seulement comments)
    df.drop( columns=df.keys()[df.isna().sum(0) == len(df)], inplace=True )

    df.to_pickle( r'{:}{:}.pkl'.format(path, filename), compression=compression)
else:
    df = pd.read_pickle( r'{:}{:}.pkl'.format(path, filename), compression=compression )

del compression

In [ ]:
df.head()

In [ ]:
df.info()

# Nettoyage

## Initial Filtering : contexte = usage non résidentiel

value counts des 'BuildingType'

In [ ]:
print(df.shape)
display( df['BuildingType'].value_counts() )

Suppression des Mutlifamily

In [ ]:
print('initial DataFrame shape:', df.shape )
sr_loc = df['BuildingType'].str.contains('multifamily')

fig, ax = plt.subplots( figsize=(16*cm,8*cm))
ax.plot( df.loc[sr_loc,'Longitude'], df.loc[sr_loc,'Latitude'], 'ro', markersize=3, label='residential', zorder=2 )
ax.plot( df.loc[~sr_loc,'Longitude'], df.loc[~sr_loc,'Latitude'], 'bo', markersize=3, label='others', zorder=1 )
ax.legend()
ax.axis('equal')
ax.set_title('Buildind locations')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

sr_loc = sr_loc[sr_loc]
df.drop( index=sr_loc.index, inplace=True )
print('DataFrame shape:', df.shape )
display( df['BuildingType'].value_counts() )

Définition d'une fonction pour déterminer les éléments correspondant à du `housing`

In [ ]:
def is_housing( sr ):
    sr_loc = (sr.str.contains('hous|multifamily', na=False)) \
        & (~sr.str.contains('warehouse|courthouse', na=True))
    return sr_loc

Vérification et suppression à partir du `PrimaryPropertyType`

In [ ]:
var = 'PrimaryPropertyType'
sr_loc = is_housing(df[var])
display( df.loc[sr_loc, var].value_counts() )
df.drop( index=sr_loc[sr_loc].index, inplace=True )
print('DataFrame shape:', df.shape )

1. Vérification et suppression pour le `LargestPropertyUseType`
2. Vérification des LargestPropertyUseType

In [ ]:
vars = ['LargestPropertyUseType',
        'SecondLargestPropertyUseType',
        'ThirdLargestPropertyUseType']


sr_loc = is_housing( df[vars[0]] )

# sr_loc = is_housing( df[vars[1]] )
# for var in vars[2:]:
#     sr_loc = sr_loc | is_housing( df[var] )

# display( df['LargestPropertyUseType'].value_counts() )
# display( df['SecondLargestPropertyUseType'].value_counts() )
# display( df['ThirdLargestPropertyUseType'].value_counts() )

print('DataFrame shape:', df.shape )
display( f'housing find in {sr_loc.sum()} elements' )
display( df.loc[sr_loc, vars] )

df.drop( index=sr_loc[sr_loc].index, inplace=True )

print('DataFrame shape:', df.shape )

del vars, var

## Colonnes Outlier et DefaultData

In [ ]:
def value_counts( sr ):
    value_counts = sr.value_counts()
    return pd.DataFrame( {'count': value_counts.values, 
                        '%':value_counts.values*(100/len(sr)) },
                        index=value_counts.index)

print('Outlier:')
display( value_counts( df['Outlier'] ) )
print('DefaultData:')
display( value_counts( df['DefaultData'] ) )

Mapping des "Outliers"

In [ ]:
df['Outlier'] = df['Outlier'].map( {'low outlier':-1, 'high outlier':1}).fillna(0).astype('int')

## Vérifications doublons

In [ ]:
print("Number of duplicated: {:}".format( df['OSEBuildingID'].duplicated().sum() ) )

## Variables "inutiles"

Vérification des variables inutiles: création d'une liste "à supprimer" (suppression faite à la fin)

In [ ]:
vars_to_delete = ['OSEBuildingID', 'PropertyName',
        'TaxParcelIdentificationNumber',
        # 'CouncilDistrictCode',
        'Address', 'ListOfAllPropertyUseTypes']
# df.drop( columns=vars, inplace=True )

In [ ]:
vars_to_check = ['DataYear', 'City', 'State']

for var in vars_to_check:
    if not var in df.keys() :
        print( f'{var} not in DataFrame')
        continue
    sr = df[var].value_counts()
    display(sr)
    if len(sr) < 2:
        # df.drop( columns=var, inplace=True )
        vars_to_delete.append( var )
        print( f'{var} added to the drop list')

## Variable liée à la date
Une variable "AgeOfBuilding(s)" est créée à partir des variables "DataYear" et "YearBuilt".

La variables "DataYear" est remplacée puisque considérée comme inutile (une unique valeur: 2016).

In [ ]:
df[['DataYear', 'YearBuilt']].info()
df['DataYear'] = df['DataYear'] - df['YearBuilt']
df.rename( columns={'DataYear':'AgeOfBuilding(s)'}, inplace=True )
vars_to_delete.append( 'YearBuilt' )

## Variable de localité

In [ ]:
mask = 'Neighborhood'
display( df[mask].value_counts() )

mapping "delridge neighborhoods" $\rightarrow$ "delridge"

In [ ]:
df.loc[ df[mask] == 'delridge neighborhoods', mask ] = 'delridge'
display( df[mask].value_counts() )

In [ ]:
df['ZipCode'].value_counts()

On note une corrélation partielle entre le ZipCode et le Neighborhood.
Cependant, certains ZipCode ne contiennent qu'une seule valeur.

In [ ]:
X = "ZipCode"
Y = "Neighborhood"

# display( df[X].value_counts())

tmp = df[[X,Y]].dropna()
tmp[X] = tmp[X].astype(int)
cont = tmp.pivot_table(index=X,columns=Y,aggfunc=len,margins=True,margins_name="Total")
col_index = cont.fillna(0).values[:-1,:-1].argmax( 1 )
# display( col_index )
col_index = [id for i, id in enumerate(col_index) if not id in col_index[:i]] + [cont.shape[1]-1]
# display( col_index )
cont = cont.iloc[:, col_index]
display(cont)


tx = cont.loc[:,["Total"]]
ty = cont.loc[["Total"],:]
indep = tx @ ty / len(df)



# c = cont.fillna(0) # On remplace les valeurs nulles par 0
c = cont
measure = (c-indep)**2/indep
xi_n = measure.sum().sum()
table = measure/xi_n
sns.heatmap(table.iloc[:-1,:-1],annot=c.iloc[:-1,:-1], fmt='.0f')

In [ ]:
vars_to_delete += ['ZipCode']

On affiche les zones avec un scatter plot. Le package folium peut être utilsié pour afficher une carte interactive.

In [ ]:
import folium
import folium.plugins

fig = folium.Figure(width=400, height=500)
seattle_map = folium.Map( location=[df['Latitude'].mean(), df['Longitude'].mean()], zoom_start=11).add_to(fig)

categories = df['Neighborhood'].value_counts().index
# dict_color = { categ:color for categ,color in zip( categories, sns.color_palette( n_colors=categories.size ) ) }
folim_colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']
# 'beige', 'white', 'pink',
dict_color = { categ:color for categ,color in zip( categories, folim_colors ) }

for lat, lon, categ in zip(df['Latitude'], df['Longitude'], df['Neighborhood']):
    # print('color:', dict_color[categ])
    folium.CircleMarker(
        [lat, lon],
        radius=5,
        popup = ( str(categ)
        #          'var 2 : ' + str(bike) + '<br>'
        #          'var 3: ' + str(traffic) +'%'
                ),
        color=dict_color[categ],
        fill_color=dict_color[categ],
        fill=True,
        fill_opacity=0.9
        ).add_to(seattle_map)
seattle_map

# fig, ax = plt.subplots( figsize=(18*cm,12*cm))
# scttr = sns.scatterplot( data=df, x='Longitude', y='Latitude', hue='Neighborhood', ax=ax )
# ax.axis('equal')
# ax.set_title('Buildind locations')
# ax.set_xlabel('Longutide')
# ax.set_ylabel('Latitude')

# # for legend text
# plt.setp(scttr.get_legend().get_texts(), fontsize='7') 
 
# # for legend title
# plt.setp(scttr.get_legend().get_title(), fontsize='10') 
# plt.show()


## Recherche et gestion des NaN
### Détection des NaN
- Pour les Second & Third LargestPropertyUseType : uniquement 1 ou 2 utilisation du bâtiment $\rightarrow$ à gerer lors du hot encoding
- Pour le YearsENERGYSTARCertified : pas de certification
- Pour le ENERGYSTARscore : essayer de le modéliser ?

In [ ]:
sum_isna = df.isna().sum()
print( 'sum isna > 0:' )
display( sum_isna[sum_isna > 0])

In [ ]:
if False:
    ax = msno.bar( df )
    ax = msno.matrix( df.sort_values( by=['SecondLargestPropertyUseType','ThirdLargestPropertyUseType'] ) )

Il y a clairement des lignes avec un gros manque d'informations -> à enlever
- sur les energy : fillna(0) + sum(1) == 0 -> drop
- LargestPropertyUse : à remplir à partir du PrimaryPropertyType 

In [ ]:
energy_keys = ['SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)',
'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)',
'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)',
'SteamUse(kBtu)', 
'Electricity(kWh)', 'Electricity(kBtu)',
'NaturalGas(therms)', 'NaturalGas(kBtu)',
'TotalGHGEmissions', 'GHGEmissionsIntensity']

property_use_keys = [ 'PrimaryPropertyType', 'ListOfAllPropertyUseTypes', 'LargestPropertyUseType']

others = ['NumberofBuildings']

keys = property_use_keys + others + energy_keys

sr_loc = df[keys].isna().sum(1) > 0
print('Entries with empty cells:')
display( df.loc[sr_loc, keys])

# DROP DATA WITHOUT ENERGY INFORMATION / CONSUMPTION
indexes = (df[energy_keys].fillna(0.).sum(1) == 0.)
indexes = indexes[indexes].index
print('indexes to drop:', indexes.values)
print('DataFrame shape:', df.shape)
df.drop( index=indexes , inplace=True )

print('Entries with empty cells (after drop):')
sr_loc = df[keys].isna().sum(1) > 0
display( df.loc[sr_loc, keys])
print('DataFrame shape:', df.shape)

### Remplissage des NaN
On regarde les éléments sans LargestPropertyUseType

In [ ]:
vars = ['PrimaryPropertyType', 'ListOfAllPropertyUseTypes',
        'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'PropertyGFABuilding(s)',
        'PropertyGFAParking', 'PropertyGFATotal' ]
sr_loc = (df['LargestPropertyUseType'].isna())
display( df.loc[sr_loc,vars])

In [ ]:
display( df.loc[ df['PrimaryPropertyType']=="self-storage facility", 'LargestPropertyUseType'].value_counts() )
display( df.loc[ df['PrimaryPropertyType'].str.contains('office'), 'LargestPropertyUseType'].value_counts() )

À priori on peut replacer les valeurs manquantes de "LargestPropertyUseType(GFA)" par la "PrimaryPropertyType" / le "PropertyGFABuilding(s)".

Note: si présence de "office" dans le "PrimaryPropertyType", alors la valeur est "office" (pas de distinctions pour les "LargestPropertyUseType")

In [ ]:
for index in sr_loc[sr_loc].index:
    Primary = df.at[index, 'PrimaryPropertyType']
    df.at[index, 'LargestPropertyUseTypeGFA'] = df.at[index, 'PropertyGFABuilding(s)']
    if 'office' in Primary:
        df.at[index, 'LargestPropertyUseType'] = 'office'
        continue
    df.at[index, 'LargestPropertyUseType'] = Primary

On vérifie le résultat:

In [ ]:
display( df.loc[sr_loc,vars] )

In [ ]:
sum_isna = df.isna().sum()
print( 'sum isna > 0:' )
display( sum_isna[sum_isna > 0])

Remplissage du ZipCode à partir de la longitude et de la latitude: toutes les données étant à Seattle, les variables `Longitude` et `Latitude` peuvent être utilisée comme (`x`,`y`).

Si il y avait plus d'écarts entre les positions, possibilité d'utilisé le package `haversine`.

In [ ]:
sr_loc = df['ZipCode'].isna()
for index in sr_loc[sr_loc].index:
    x,y = df.loc[index, ['Longitude', 'Latitude']].values
    for value in df['Longitude']:
        if isinstance( value, str):
            print('value:', value)
    df['Longitude'].values-x
    df['Latitude'].values-y
    argsort = ((df['Longitude'].values-x)**2 + (df['Latitude'].values-y)**2).argsort()
    for i in argsort[1:]: # neglect the first as it corresponds to the current index
        if np.isnan( df['ZipCode'].iloc[i] ) :
            continue
        df.at[index, 'ZipCode'] = df['ZipCode'].iloc[i]
        break 
    # print('\nx,y:', x,y, '\nx2,y2:', df[['Longitude','Latitude']].iloc[i,:].values )

df['ZipCode'] = df['ZipCode'].astype(int)

sum_isna = df.isna().sum()
print( 'After procees sum isna > 0:' )
display( sum_isna[sum_isna > 0])

## Recherches à partir des variables "DefaultData" et "ComplianceStatus"

In [ ]:
fig, axs = plt.subplots( ncols=2, figsize=(24*cm,12*cm))
for ax, key in zip( axs, ['DefaultData', 'ComplianceStatus']):
    df[key].value_counts(normalize=True).plot( kind='pie', ax=ax, autopct='%.1f%%' )
    ax.set_title( key)
    ax.set_ylabel('')

On peut se poser la question de l'intérêt de supprimer certaines entrées.

Dans l'idée, les données seront mesurées pour les nouveaux bâtiments, donc autant enlever les entrées avec des valeurs imputées.

In [ ]:
sr_loc = df['DefaultData'] | ( df['ComplianceStatus'].str.contains( '|'.join( ['missing', 'error'] ) ) )
df.drop( index=sr_loc[sr_loc].index ,inplace=True )

***
# Vérifications des données d'entrée

In [ ]:
df.columns

## Value_counts des variables

In [ ]:
print('DataFrame shape:', df.shape )

In [ ]:
vars = ['BuildingType', 'PrimaryPropertyType',
       'Neighborhood', 'NumberofBuildings',
       'NumberofFloors', 'LargestPropertyUseType',
        'SecondLargestPropertyUseType',
        'ThirdLargestPropertyUseType',
       ]
for var in vars:
    print( f'{var} value_counts:')
    display( df[var].value_counts() )

Il semble incohérent d'avoir 0 bâtiments. Idem pour "number of floors" qui, en anglais, prend en compte le rez-de-chaussé.

In [ ]:
for X in ['NumberofBuildings', 'NumberofFloors']:
    df.loc[ df[X]==0, X ] = 1
df['NumberofBuildings'] = df['NumberofBuildings'].astype(int)

## Diverses vérifications

### Number of floors
On regarde les number of floors pour voir si il y a des valeurs aberrantes

In [ ]:
X, Y = 'PropertyGFABuilding(s)', 'NumberofFloors'

fig, ax = plt.subplots( figsize=(12*cm,8*cm) )
ax.plot( df[X], df[Y], 'bo', markersize=3 )

ax.set_ylabel( 'Number of floors' )
ax.set_xlabel( 'Property GFA buildings (sf)' )

index_100 = df[Y] > 90
index_100 = index_100[index_100].index.values[0]

_ = ax.annotate( 'valeur aberrante :\n{:}'.format( 
                df.at[index_100, 'PropertyName'] ),
                xy=[0, 99], xytext=[2.1e6, 82],# ha='left', va='center',
                arrowprops=dict(shrink=0.05) )

index_high_GFA = df[X] > 8e6
index_high_GFA = index_high_GFA[index_high_GFA].index.values[0]

_ = ax.annotate( 'valeur atypique :\n{:}'.format( 
                df.at[index_high_GFA, 'PropertyName'].replace(' - ', '\n') ),
                xy=[df.at[index_high_GFA , X], 1], xytext=[2.3e6, 32],# ha='left', va='center',
                arrowprops=dict(shrink=0.05) )

On assigne la valeur la plus utilisée pour NumberofFloors: 1, sachant que la surface est relativement faible.

In [ ]:
df.at[index_100, Y] = 1

### Number of Buildings

Certains `NumberofBuildings` sont particulièrement élevés, mais à priori OK.

In [ ]:
mask = 'NumberofBuildings'
df.loc[ df[mask] > 5, :].sort_values( by=mask )

### PorpertyGFA
Présence d'outliers ?

`university of washington - seattle campus` et `entire campus` correspondent à des valeurs `atypique` mais non aberrantes

In [ ]:
mask = 'PropertyGFATotal'
describe = df[mask].describe()
print(mask)
display(describe)

print('Q3 + 1.5*IQ = {:.3e}'.format( describe['75%'] 
            + 1.5*(describe['75%']-describe['25%']) ) )

df_sorted = df.sort_values( by=mask, ascending=False ).iloc[:5,:]
display( df_sorted )

Les 2 universités sont des valeurs atypiques.

In [ ]:
if False:
    print('DataFrame shape:', df.shape)
    display( df_sorted.iloc[:2,:] )
    df.drop( index=df_sorted.index[:2], inplace=True )
    print('DataFrame shape:', df.shape)

### GFA parking
Beaucoup de bâtiments semblent ne pas avoir de parkings

In [ ]:
mask = 'PropertyGFAParking'
describe = df[mask].describe()
print('Q3 + 1.5*IQ = {:.3e}'.format( describe['75%'] 
            + 1.5*(describe['75%']-describe['25%']) ) )
display(describe.T)
df.sort_values( by=mask, ascending=False ).iloc[:5,:]

## Categories "LargestPropertyUseType"

Liste des catégories

In [ ]:
df['LargestPropertyUseType'].value_counts().add(
    df['SecondLargestPropertyUseType'].value_counts(), fill_value=0 ).add(
    df['ThirdLargestPropertyUseType'].value_counts(), fill_value=0 ).astype(int).sort_values()

Affichage de quelques catégories

In [ ]:
df.loc[ df['LargestPropertyUseType'].str.contains('other - entertainment/public assembly', na=False), 'PropertyName']

In [ ]:
masks = ['LargestPropertyUseType','SecondLargestPropertyUseType', 'ThirdLargestPropertyUseType']
for mask in masks:
    display( df.loc[df[mask].str.contains('recreation', na=False), mask].value_counts() )

Mapping pour réduire le nombre de catégories

In [ ]:
categories_keys = [ ('store', ['wholesale', 'mall', 'store', 'dealership', 'distribution center'] ),
                    ('utility', ['fire', 'utility', 'police', 'courthouse', 'prison', 'bank']),
                    ('restaurant', ['food', 'restaurant']),
                    ('residential - hotel', ['residential', 'housing', 'hotel', 'dormitory']),
                    ('education', ['school', 'education', 'university']),
                    ('medical', ['care', 'hospital']),
                    ('office', ['financial office']),
                    ('entertainment - public assembly', ['theater','nightclub',
                                'recreation', 'swimming', 'performing arts',
                                'library','museum','meeting hall']),
                    ('lifestyle center', ['lifestyle', 'fitness']),
                    ('science', ['technology', 'laboratory']),
                    ('services', ['services']),
                    ('industrial', ['industrial', 'refrigerated']),
             ]
for (category,keys) in categories_keys :
    joint_keys = '|'.join(keys)
    for mask in masks:
        sr_loc = df[mask].str.contains( joint_keys , na=False )
        df.loc[ sr_loc, mask ] = category


value_counts = df['LargestPropertyUseType'].value_counts().add(
    df['SecondLargestPropertyUseType'].value_counts(), fill_value=0 ).add(
    df['ThirdLargestPropertyUseType'].value_counts(), fill_value=0 ).astype(int).sort_values()

print( f'{len(value_counts)} categories')
display( value_counts )

In [ ]:
fig, ax = plt.subplots( figsize=(25*cm,25*cm) )
value_counts.plot( kind='pie', ax=ax )
fig.tight_layout()

## Year-built -> categories ?
à priori pas de relation directe évidente -> on peut séparer en plusieurs groupes

In [ ]:
X, Y = 'YearBuilt', 'SiteEUIWN(kBtu/sf)'

print('min:', df[X].min(), 'max:', df[X].max() )
print( np.arange( 1900, 2020, 20 ) )

bins_yearbuilt = [1900, 1920, 1934, 1943, 1960, 1980, 2000, 2020]
df['YearBuiltCateg'] = np.digitize( df[X], bins_yearbuilt )
display( df[[X, 'YearBuiltCateg']].sample(10) )
display( df[[X, 'YearBuiltCateg']].head(10) )

x = df[X].values.copy()
x.sort()
diff_x = x[1:]-x[:-1]
b_break = diff_x > 1
print( 'breaks begin at:', x[:-1][ b_break ])
print( 'breaks end at:', x[1:][ b_break ])

fig, ax = plt.subplots( figsize=(12*cm,6*cm) )
ax.plot( x[1:], diff_x, 'bo', markersize=3 )
ax.plot( x[1:][ b_break ], diff_x[ b_break ], 'ro', markersize=3 )
ax.set_xlabel(X)
ax.set_ylabel( 'year difference' )

fig, ax = plt.subplots( figsize=(12*cm,8*cm) )
ax.plot( df[X], df[Y], 'bo', markersize=3 )
ax_twinx = ax.twinx()
ax_twinx.plot( df[X], df['YearBuiltCateg'], 'yo', markersize=3 )


ax.set_xlabel( X )
ax.set_ylabel( Y )
fig.tight_layout()

***
# 4. Target

## 4.1 Vérifications et gestion des données

Vérification de conversion kWh -> Btu : OK

In [ ]:
X, Y = 'Electricity(kWh)', 'Electricity(kBtu)'
coef = 3.412142 # in [Btu] / [Wh]

lr = linear_model.LinearRegression()
lr.fit( df[X].values.reshape(-1,1), df[Y] )
print( f'estmated coefficient: {lr.coef_[0]:.3f}, theoretical: {coef:}' )

fig, ax = plt.subplots( figsize=(8*cm,8*cm))
ax.plot( df[X], df[Y], 'bo' )
ax.set_xlabel(X)
ax.set_ylabel(Y)
fig.tight_layout()

vars_to_delete.append( X )

Vérification de conversion therms -> Btu : OK

In [ ]:
X, Y = 'NaturalGas(therms)', 'NaturalGas(kBtu)'
coef = 1e2 # in [kBtu]/[therms]

lr = linear_model.LinearRegression()
lr.fit( df[X].values.reshape(-1,1), df[Y] )
print( f'estmated coefficient: {lr.coef_[0]:.3f}, theoretical: {coef:}' )

fig, ax = plt.subplots( figsize=(8*cm,8*cm))
ax.plot( df[X], df[Y], 'bo' )
ax.set_xlabel(X)
ax.set_ylabel(Y)
fig.tight_layout()

vars_to_delete.append( X )

Pair plot des données énergie: les consommations sont principalement électrique, avec un partage avec du nas naturel, et relativement peu de vapeur. 

In [ ]:
energies = ['Electricity(kBtu)', 'SteamUse(kBtu)', 'NaturalGas(kBtu)']
tmp = df[energies].copy()
for X in energies:
    tmp[X] = 100* tmp[X] / df['SiteEnergyUse(kBtu)']
    tmp.rename( columns={X:X[:-6] + ' (%)'}, inplace=True)

g = sns.pairplot( data=tmp, plot_kws={'s':6}, diag_kws={'bins':60} )
g.fig.tight_layout()

Vérification de la somme des ressources utilisées.

`SiteEnergyUse(kBtu)`: The annual amount of energy consumed by the property from all sources of energy.

Certaines valeurs diffèrent mais la proportion semble assez faible.<br>Cette différence est possiblement due à de l'électricity générée par des panneaux solaires par exemple (un bâtiment à une valeur de Electricity négative, une recherche rapide indique qu'il s'agit d'un bâtiments écologique avec des panneaux solaires sur le toit)

Les valeurs 'Electricity(kBtu)', 'SteamUse(kBtu)', 'NaturalGas(kBtu)' sont aussi mieux corrélées à la valeur 'SiteEnergyUse(kBtu)' qu'à la valeur WeatherNormalized.

In [ ]:
X, Y = energies, 'SiteEnergyUse(kBtu)'

display( df[X].describe() )

X_values = df[X].sum(1)

print('X_values:')
display(X_values.describe())

print('{:} sumed values equals 0'.format( (X_values==0).sum() ) )

diff = X_values - df[Y]
diff_ratio = diff / df[Y].abs() * 100


print('\nBulding with negative Electricity:')
display(df.loc[df['Electricity(kBtu)'] < 0 , :] )

print('DataFrame shape:', df.shape)
print('{:} values differs more than 1%'.format( (diff_ratio.abs() > 0.01).sum() ))

print('{:} values differs more than 30%'.format( (diff_ratio.abs() > 0.3).sum() ))
print('{:} values corresponds to sum energies > 1.1 * SiteEnergy'.format( (diff_ratio > 0.1).sum() ))

lr = linear_model.LinearRegression()
lr.fit( X_values.values.reshape(-1,1), df[Y].values )
print( f'\nEstmated coefficient: {lr.coef_[0]:.3f}, ideal: {1:}' )

display( df.loc[ diff_ratio > 0.1, :] )

display( pd.DataFrame( {Y:df[Y], 'sum ressources':X_values} ).describe().T )

fig, ax = plt.subplots( figsize=(12*cm,8*cm))
ax.plot( X_values, df[Y], 'bo', markersize=2 )
ax.plot( [0, 8e8], [0,8e8], 'r', label='bisector', zorder=1)
ax.legend()
ax.set_xlabel(r'$\sum$ steam,elec.,gas')
ax.set_ylabel(Y)

# ax_twinx = ax.twinx()
# ax_twinx.plot( X_values, diff_ratio, 'ro', markersize=4 )
# ax_twinx.set_ylabel('difference (%)', color='r')

fig.tight_layout()

Le siteEnergyUse est recalculé puisque des erreurs sont visibles

In [ ]:
# pour le X_values == 0 -> considérons une ressource purement électrique puisque plus probable en regardant le pairplot plus haut)
loc = X_values == 0
df.loc[loc,'Electricity(kBtu)'] = df.loc[loc, Y]

df[Y] = df[energies].sum(1)

X_values = df[energies].sum(1)
fig, ax = plt.subplots( figsize=(12*cm,8*cm))
ax.plot( X_values, df[Y], 'bo', markersize=2 )
ax.plot( [0, 8e8], [0,8e8], 'r', label='bisector', zorder=1)
ax.legend()
ax.set_xlabel(r'$\sum$ steam,elec.,gas')
ax.set_ylabel(Y)

fig.tight_layout()

SiteEnergyUse(WN) 

In [ ]:
X, Y = 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)'

x,y = df[[X,Y]].dropna().values.T

loc = y != 0.

lr_WN = linear_model.LinearRegression()
lr_WN.fit( x[loc].reshape(-1,1), y[loc] )
print( f'estmated coefficient: {lr_WN.coef_[0]:.3f}' )

fig, ax = plt.subplots( figsize=(10*cm,8*cm))
ax.plot( x, y, 'bo', markersize=3, zorder=2 )
ax.plot( [0,5e8], [0,lr_WN.coef_[0]*5e8], 'r', zorder=1 )
ax.set_xlabel(X)
ax.set_ylabel(Y)
fig.tight_layout()

Certaines valeurs sont à 0. ou semblent ne pas correspondre -> recalcul de la somme des énergies et application du coefficient de la régression linéaire

In [ ]:
sr_loc = (df[Y] == 0.) | (df[Y].isna())
print('nombre de 0 | NaN:', sr_loc.sum())
display( df.loc[sr_loc,:] )

df[Y] = lr_WN.predict( df[X].values.reshape(-1,1) ).ravel()

sr_loc = (df[Y] == 0.) | (df[Y].isna())
print('nombre de 0 | NaN:', sr_loc.sum())

fig, ax = plt.subplots( figsize=(10*cm,8*cm))
ax.plot( df[X], df[Y], 'bo', markersize=3, zorder=2 )
ax.plot( [0,8e8], [0,lr_WN.coef_[0]*8e8], 'r', zorder=1 )
ax.set_xlabel(X)
ax.set_ylabel(Y)
fig.tight_layout()


Données par unité de surface
- `SiteEUI(kBtu/sf)` est basée sur les factures
- `SourceEUI(kbtu/sf)` : "the annual energy used to operate the property, including losses from generation, transmission, & distribution."

In [ ]:
X, Y = 'SiteEUI(kBtu/sf)', 'SourceEUI(kBtu/sf)'

lr = linear_model.LinearRegression()
lr.fit( df[X].values.reshape(-1,1), df[Y] )
print( f'estmated coefficient: {lr.coef_[0]:.3f}' )

fig, ax = plt.subplots( figsize=(8*cm,8*cm))
ax.plot( df[X], df[Y], 'bo', markersize=3 )

x = np.array([0, df[X].max()]).reshape(2,1)
ax.plot( x, lr.predict(x), 'r' )
ax.set_xlabel(X)
ax.set_ylabel(Y)
fig.tight_layout()

On regarde à quoi correspondent les surfaces

Il y a clairement des incohérences entre les `LargestPropertyUseTypeGFA` et (`PropertyGFATotal`, `PropertyGFABuilding(s)`, `PropertyGFAParking`, `NumberofFloors`)

In [ ]:
x = df[['LargestPropertyUseTypeGFA',
        'SecondLargestPropertyUseTypeGFA',
        'ThirdLargestPropertyUseTypeGFA']].fillna(0.).sum(1)


Y = 'PropertyGFATotal'
y = df[Y]


fig, ax = plt.subplots( figsize=(12*cm,8*cm))
ax.plot( x, y, 'bo', markersize=2 )
ax.plot( [0, y.max()], [0, y.max()], 'r', zorder=0, label='bisector' )
ax.legend()
ax.set_xlabel(r'$\sum$ PrincipalUseTypeGFA')
ax.set_ylabel(Y)

fig.tight_layout()

On regarde la cohérence entre le SiteEnergyUseWN(kBtu/sf) et le SiteEUIWN(kBtu/sf):
- la surface utilisée pour calculer le SiteEUIWN semble être plus proche de la somme des GFA use tye
- comme on souhaite modéliser la consommation et les émissiones `totales`, les variables surfaciques seront recalculée à partir du GFABuilding(s)

In [ ]:
print( df.keys().tolist())
fig, axs = plt.subplots( ncols=2,nrows=2, figsize=(20*cm,18*cm))

for i, (srfc, srfc_label) in enumerate( zip( [y,x], ['GFA total', r'$\sum Principal use type GFA$'] ) ):
    for j, (X,Y) in enumerate([ ('SiteEnergyUseWN(kBtu)', 'SiteEUIWN(kBtu/sf)'), ('SiteEnergyUse(kBtu)', 'SiteEUI(kBtu/sf)')]) :
        ax = axs[j][i]
        # X_values = df[X] / y
        X_values = df[X] / srfc
        diff = X_values - df[Y]

        sr_drop = (X_values.isna()) | ( df[Y].isna())

        lr = linear_model.LinearRegression()
        lr.fit( X_values.loc[~sr_drop].values.reshape(-1,1), df.loc[~sr_drop, Y] )
        # print( f'\nEstmated coefficient: {lr.coef_[0]:.3f}, ideal: {1:}' )

        display( pd.DataFrame( {Y:df[Y], 're-calculated':X_values} ).describe().T )

        ymax = df[Y].max()
        xmax = X_values.max()
        ax.set_title( f'coef.: {lr.coef_[0]:.3f}' )
        ax.plot( [0,xmax], [0,xmax], 'r', label='ideal ratio' )
        ax.plot( [0,xmax], lr.predict( np.array([0,xmax]).reshape(-1,1) ), 'y', label='estimated ratio' )
        ax.plot( X_values, df[Y], 'bo', markersize=2 )
        ax.set_xlabel( X + f'\n/\n({srfc_label})')
        ax.set_ylabel(Y)
        ax.legend()

        # ax_twinx = ax.twinx()
        # ax_twinx.plot( X_values, np.abs(diff) / X_values * 100, 'ro', markersize=4 )
        # ax_twinx.set_ylabel('difference (%)', color='r')

        fig.tight_layout()

In [ ]:
vars_to_delete += df.columns[df.columns.str.contains('/sf|Intensity')].tolist()

In [ ]:
x,y = df[['PropertyGFABuilding(s)', 'SiteEnergyUseWN(kBtu)']].values.T

values = y / x

# affichage des valeurs élevée
display( df.iloc[ values.argsort()[-10:] , : ] )


fig, ax = plt.subplots( figsize=(12*cm,8*cm) )
ax.hist( y/x, bins=30 )
ax.set_xlabel( 'SiteEnergyUseWN / GFA Buildind(s) (kBtu/sf)')

Le TotalGHGEmissions est calculé en utilisant :
> - Light's 2015 emissions factor of 52.44 lbs CO2e/MWh until the 2016 factor is available.
> - Enwave steam factor = 170.17 lbs CO2e/MMBtu.
> - Gas factor sourced from EPA Portfolio Manager = 53.11 kg CO2e/MBtu.

La variables `TotalGHGEmissions` est exprimée en `metric tons of carbon dioxide equivalent`.

Estimer cette variable revient à estimer l'`énergie consommée` connaissant les `proportions d'énergies utilisée`.

In [ ]:
y = df['TotalGHGEmissions']

def energies_to_emissions( elec_kBtu, steam_kBtu, gas_kBtu):
    return 1e-3 *( # conversion kg->tons
            0.4535924*( # conversion lbs-> kg
                (52.44e-3/3.412142) * elec_kBtu # conversion kBtu->kWh->lbs
                + 170.17e-3 * steam_kBtu) # conversion kBtu->lbs
                + 53.11e-3 * gas_kBtu ) # coonversion kBtu->kg

# x =  0.4535924*(52.44e-3 * df['Electricity(kWh)'] + 170.17e-3 * df['SteamUse(kBtu)']) + 53.11e-3 * df['NaturalGas(kBtu)']
# x = x*0.001 # conversion en tonne

# x = df[energies].values.T*1e-3

x = energies_to_emissions( *(df[energies].values.T) )
lr = linear_model.LinearRegression()
lr.fit( x.reshape(-1,1), y )


vmax = max( x.max(), y.max() )
fig, ax = plt.subplots( figsize=(12*cm,8*cm) )
ax.plot( [0, vmax], [0, vmax], 'r', label='bisector')
ax.plot( x, y, 'bo', markersize=2 )
ax.set_xlabel( 'valeur calculée à partir des\nénergies consommées (tons CO2)')
ax.set_ylabel( 'TotalGHGEmissions (tons CO2)')
ax.legend()

print('estimated coefficient: {:.3e}'.format( lr.coef_[0]))

Avec la valeur `WeatherNormalized` pour le  `SiteEnergyUse` calculer la valeur `WeatherNormalized` pour le `TotalGHGEmissions` sera plus coherent (voir section Feature des modèles).

In [ ]:
targets = ['SiteEnergyUseWN(kBtu)', 'TotalGHGEmissions']

In [ ]:
df[targets].info()

for target in targets:
    sr_0 = (df[target] == 0) | (df[target].isna())
    print( f'\nnumber of elements {target} == 0 | isna:', (sr_0).sum())
# display( df.loc[sr_0,:])

Des emissions sont à 0, ce qui n'est pas cohérent

In [ ]:
df.loc[sr_0,:]

Statistiquement, la valeur manquante est fort probablement Electrique

In [ ]:
df.loc[sr_0,'TotalGHGEmissions'] = energies_to_emissions( *( df.loc[sr_0, energies].values.T ))

df.loc[sr_0,:]

In [ ]:
# power_inv_targets = [6, 6]
# for target, power_inv in zip( targets, power_inv_targets ):
for target in targets:
    y = df[target].dropna()

    display( y.describe() )

    # y_pow = (y+1.-y.min())**(1/power_inv)

    # print( ( y + (1-y.min()) ).min() )
    y_log = np.log( y + (1-y.min() ) )


    fig, axs = plt.subplots( ncols=2, figsize=(18*cm,8*cm) )
    n, bins, _ = axs[0].hist( y, bins=50 )
    n, bins, _ = axs[1].hist( y_log, bins=50 )
    # n, bins, _ = axs[2].hist( y_pow, bins=50 )
    axs[0].set_xlabel( 'values' )
    axs[1].set_xlabel( f'log( values )' )
    # axs[2].set_xlabel( f'values ** (1/{power_inv})' )

    axs[0].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y), st.kurtosis(y) ) )
    axs[1].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y_log), st.kurtosis(y_log) ) )
    # axs[2].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y_pow), st.kurtosis(y_pow) ) )

    fig.suptitle( target )
    fig.tight_layout()

    log_target_bins_center = (bins[:-1] + bins[1:]) * 0.5 

Il est probablement plus intéressant d'estimer le SiteEnergyUse(kBtu) divisé par le PropertyGFABuilding(s).<br>
Le feature engineering est à adapté en conséquence.

In [ ]:
y = df[targets[0]] / df['PropertyGFABuilding(s)']
y_label = targets[0] + ' / GFA Building(s)'

display( y.describe() )

y_log = np.log( y + (1-y.min() ) )


fig, axs = plt.subplots( ncols=2, figsize=(18*cm,8*cm) )
n, bins, _ = axs[0].hist( y, bins=50 )
n, bins, _ = axs[1].hist( y_log, bins=50 )
# n, bins, _ = axs[2].hist( y_pow, bins=50 )
axs[0].set_xlabel( 'values' )
axs[1].set_xlabel( f'log( values )' )
# axs[2].set_xlabel( f'values ** (1/{power_inv})' )

axs[0].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y), st.kurtosis(y) ) )
axs[1].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y_log), st.kurtosis(y_log) ) )
# axs[2].set_title( r'$\gamma_1$: {:.3f}, $\gamma_2$: {:.3f}'.format( st.skew(y_pow), st.kurtosis(y_pow) ) )

fig.suptitle( y_label )
fig.tight_layout()

log_target_bins_center = (bins[:-1] + bins[1:]) * 0.5 

## 4.2 Scatter plots

In [ ]:

Xs = ['PropertyGFABuilding(s)', 'LargestPropertyUseTypeGFA']
hues = ['PrimaryPropertyType', 'LargestPropertyUseType']

# df[target]

Y = targets[0]

sr_loc = (df[Y] != 0)
for X in Xs:
    sr_loc = sr_loc & (df[X]!=0)
tmp = df.loc[sr_loc, Xs + hues + [Y, 'Outlier']]
# tmp = df[[X,target, 'Outlier', hue]]

if True:
    # tmp[ Xs[1] ] /= tmp[ Xs[0] ]
    # sr_loc = tmp[Xs[1]] > 0.7
    sr_loc = tmp[hues[0]].str.contains('office')
    tmp = tmp.loc[sr_loc,:]


for X,hue in zip( Xs, hues ):
    g = sns.pairplot( data=tmp, vars=[X,Y], hue=hue,
        plot_kws={'s':6} )
    
    handles = g._legend_data.values()
    labels = g._legend_data.keys()
    g.fig.legend(handles=handles, labels=labels, loc=[0.2,0.65], ncol=1)

    g.fig.suptitle('linear values')
    g.fig.tight_layout()
    

    tmp[X] = np.log( tmp[X] )
    tmp[Y] = np.log( tmp[Y] )
    g = sns.pairplot( data=tmp, vars=[X,Y], hue=hue,
        plot_kws={'s':6} )
    # g.legend(bbox_to_anchor= (1.03, 1) )
    handles = g._legend_data.values()
    labels = g._legend_data.keys()
    g.fig.legend(handles=handles, labels=labels, loc=[0.35,0.65], ncol=1)
    g.legend.remove()
    g.fig.suptitle('log values')
    g.fig.tight_layout()
    
    break

Le PropertyGFABuilding(s) est fortement corrélé au SiteEnergyUseWN(kBtu) pour les offices.
Les gorupes sont différents puisque les GFA sont différents.

In [ ]:
def ANOVA( df, hue, Y, transform=None ):
    groups = df[[Y,hue]].dropna().groupby(hue)[Y]
    index = pd.MultiIndex.from_tuples( [('shape','skew'), ('shape','kurtosis'),('shapiro','statistic'), ('shapiro','p-value')] )
    df_shape = pd.DataFrame( index = index)
    names = [name for name,yi in groups]
    for name, yi in groups:
        if not transform is None:
            yi = transform( yi )
        # print('\nname:', name)
        # print( st.skew(yi) )
        # print( st.kurtosis(yi) )
        # print( st.shapiro(yi) )
        # sr = pd.Series( [  ] ).astype(float)
        shapiro = st.shapiro(yi)
        df_shape[name] = [st.skew(yi), st.kurtosis(yi), shapiro.statistic, shapiro.pvalue]
    
    print(f'ANOVA on variable {Y}')
    print('p-value is the probability to have found a particular set of observations if the null hypothesis were true (valid if p-value is high)')
    print('Shapiro null hypothesis: data has a normal distribution')
    display(df_shape.round(3))

    levene = pg.homoscedasticity( df.dropna(), dv=Y, group=hue )
    print( 'small p-value suggests that the populations do not have equal variances' )
    display( levene )

    # tpl = tuple( [df.query( f'{hue} == "{name}"')[Y].dropna() for name in names ] )
    # print( st.levene( *tpl ) )
    

    print('small p-value suggests that there is a statistically significant difference between the means of groups')
    display( pg.pairwise_tukey( dv=Y, between=hue, data=df.dropna() ) )

    # for i, (name,yi) in enumerate(groups):
    #     print('\n\n| {:} |\n\n'.format( '-'*20 ))
    #     if i+1 == len(names):
    #         break
    #     print(f'name: {name}')
    #     for j, (name_2,yi2) in enumerate(groups):
    #         if j <= i:
    #             continue
    #         print( f'\n{name} - {name_2}:' )
    #         stat, p = st.levene(yi, yi2)
    #         print( f'levene stat: {stat:.3f}, p-value: {p:.3f}' )
    #         print('std: {:.3f} - {:.3f}'.format( yi.std(), yi2.std() ) )

mask = 'SiteEnergyUseWN / GFA Building(s)(kBtu/sf)'
tmp[mask] = df[targets[0]] / df['PropertyGFABuilding(s)']
print('LINEAR VALUES')
ANOVA( tmp, 'PrimaryPropertyType', mask )
print('LOG VALUES')
ANOVA( tmp, 'PrimaryPropertyType', mask, lambda y:np.log( y + (1-y.min())) )

Les "large office" et "small- and mid-sized office" semblent correspondrent à une consommation par unité de surface similaires, mais les "medial offices" semble être différents.

><span style="color:orange"> Note: </span>
> - Les p-value des test shapiro indiquent bien que les valeurs ne sont pas de distribution normale
> - le test de levene indiquent que les variances sont suffisamment semblables

***
# Features des modèles

## Suppression des variables non utilisées

In [ ]:
print('Columns to delete:', vars_to_delete)

In [ ]:
df_annexe = df[['PropertyName']].copy() # DataFrame des variables à garder sous le coude
df.drop( columns= [col for i, col in enumerate(vars_to_delete) if (col in df.columns) and (not col in vars_to_delete[:i])], inplace=True )
vars_to_delete = []
print('DataFrame.shape', df.shape)
df.columns

Les variables 'BuildingType', 'PrimaryPropertyType' ne seront pas utilisées, car redondant par rapport aux `LargestPropertyUseTypes` 

In [ ]:
df.drop( columns=['BuildingType', 'PrimaryPropertyType'] , inplace=True )
print('DataFrame.shape', df.shape)
df.columns

In [ ]:
print('DataFrame shape:', df.shape )


mask = ['YearsENERGYSTARCertified', 'Outlier',
    'YearBuiltCateg', 'DefaultData', 'ComplianceStatus']
df_annexe[mask] = df[mask].copy()
df.drop( columns=mask, inplace=True )

print('DataFrame shape:', df.shape )
print('DataFrame annexe shape:', df_annexe.shape )

df.columns

## Corrélation des variables
Une (force) corrélation est à noter entre les variables:
- PropertyTotalGFA, PropertyGFABuilding(s) LargestPropertyUseTypeGFA et SecondLargestPropertyUseTypeGFA 
- TotalGHGEmissionIntensity et SiteEUIWN(kBtu/sf)
- Electricity(kBtu) et les variables GFA

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True
fig, ax = plt.subplots(figsize=(21*cm,18*cm))
ax = sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={'size':6}, 
                 mask=mask, center=0, cmap="coolwarm")
plt.title(f"Heatmap des corrélations linéaires\n")
plt.show()

Vérification de **multicolinéarité avec le VIF** *(Variance Inflation Factor)* : $VIF = \frac{1}{1-R^2}$

On sélectionne les variables qui ne sont pas des variables cibles

In [ ]:
variables = corr.columns.values[ ~corr.columns.str.contains( 'SiteEnergy|Emissi') ]
variables

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


tmp = df[variables].dropna()

tmp_vif = pd.DataFrame()
tmp_vif["feature"] = tmp.columns
tmp_vif["VIF"] = [variance_inflation_factor(tmp.values, i) 
                   for i in range(len(tmp.columns))]

display(tmp_vif[tmp_vif['VIF'] > 5])

Des scores VIF supérieur à 5 indiquent généralement une forte multicolinéarité. Les variables fortement corrélées peuvent dégrader les résultats des modèles.

La variable ENERGYSTARScore est mise de côté, uisque dans l'idéale, il est préférable de s'en affranchir.

In [ ]:
mask = ['ENERGYSTARScore']
df_annexe[mask] = df[mask].copy()
df.drop( columns=mask, inplace=True )

## PCA

In [ ]:
tmp = df.select_dtypes(include=[np.number])
features = [col for col in tmp.columns if not col in vars_to_delete]
print(features)

In [ ]:
X = df[features].dropna()
scaler_pca = preprocessing.StandardScaler()
X_scaled = scaler_pca.fit_transform(X) # fit and transform
idx = ["mean", "std"]
display( pd.DataFrame(X_scaled).describe().round(2).loc[idx, :] )

In [ ]:
from sklearn.decomposition import PCA
n_components = X_scaled.shape[1]
pca = PCA(n_components=n_components)

# entrainement
pca.fit(X_scaled)

In [ ]:
x_list = range(1, n_components+1)
scree = (pca.explained_variance_ratio_*100)
print('scree:', scree.round(2))
print('sum scree:', scree.sum().round(2))
fig, ax = plt.subplots( figsize=(14*cm,8*cm))
ax.bar( x_list, scree )
ax.set_xlabel("rang de l'axe d'inertie")
ax.set_ylabel("inertie (%)")
ax.set_title('Éboulis des valeurs propres')
ax.yaxis.grid( visible=False )

ax = ax.twinx()
ax.set_ylabel("inertie (%)")

ax.plot( x_list, scree.cumsum(), c='r', marker='o', label='intertie cumulée')
ax.legend()

fig.tight_layout(pad=0.2)
# tools.savefig( fig, 'Figures/PCA/ebouli.pdf')

On observe les premiers axes d'inertie

In [ ]:
pcs = pd.DataFrame( pca.components_.transpose() )
pcs.index = features
columns = [f"F{i}" for i in x_list]
pcs.columns = columns

for i in range(6):
    key = f'F{i+1:}'
    display( pcs[[key]].sort_values( key, ascending=False ).T )

display( pcs.iloc[:,:6].round(2).T ) #.sort_values(by=indexes , ascending=False) )
fig, ax = plt.subplots(figsize=(26*cm, 12*cm))
sns.heatmap(pcs.iloc[:,:6].T*100, vmin=-100, vmax=100, annot=True, cmap="coolwarm", fmt="0.0f", annot_kws={"size": 8})
# fig.tight_layout( pad=0.2 )

## Calcul de ratios

On s'intrésse à des variables rapportée à la surface totale d'un site, donc on ramène les GFA à des ratios, avec comme référence les GFABuilding(s)

In [ ]:
GFA_ref = df['PropertyGFABuilding(s)'].values.copy()

mask = ['PropertyGFATotal', 'PropertyGFABuilding(s)', 'PropertyGFAParking']
df_annexe[mask] = df[mask].copy()
df.drop( columns=mask, inplace=True )


mask2 = ['LargestPropertyUseTypeGFA', 'SecondLargestPropertyUseTypeGFA', 'ThirdLargestPropertyUseTypeGFA']
df[ mask2 ] = df[mask2].fillna(0.).values / GFA_ref.reshape(-1,1)
# df[ mask2 ] = df[ mask2 ] / df[ mask2 ].values.sum(1).reshape(-1,1)
df.rename( columns={ mask2[0]:'prop_use_1', mask2[1]:'prop_use_2', mask2[2]:'prop_use_3' }, inplace=True)
df.rename( columns={ mask2[0][:-3]:'Type_use_1', mask2[1][:-3]:'Type_use_2', mask2[2][:-3]:'Type_use_3' }, inplace=True)
df.sample(5)

Pour éviter les fuites de données, des ratios sont calculés pour les énergies

In [ ]:
df[energies] = df[energies].fillna(0.)
sr_0 = df[energies].sum(1) == 0.
print(f'Entrées avec des 0 : {sr_0.sum()}')
# display( df.loc[sr_0,:] )

df.loc[sr_0,'Electricity(kBtu)'] = 1.

# df[energies] = df[energies].values / df[energies].abs().values.sum(1).reshape(-1,1)
df[energies] = df[energies] / df['SiteEnergyUse(kBtu)'].values.reshape(-1,1)
df.rename( columns={ var:'prop_{:}'.format( var.replace('(kBtu)', '') ) for var in energies }, inplace=True )
df.sample(5)

Calcul du TotalGHGEmissions `WeatherNormalized`. Les valeurs obtenues sont bien du même ordre de grandeur que les valeurs brutes.

In [ ]:
prop_energies = [ 'prop_{:}'.format( var.replace('(kBtu)', '') ) for var in energies]
df['TotalGHGEmissionsWN'] = energies_to_emissions( *( ( df[prop_energies].values * df['SiteEnergyUseWN(kBtu)'].values.reshape(-1,1) ).T ) )
display( df[['TotalGHGEmissions','TotalGHGEmissionsWN']].describe().T )
df.head(5)

In [ ]:
mask = ['SiteEnergyUse(kBtu)', 'TotalGHGEmissions']
df_annexe[mask] = df[mask].copy()
df.drop( columns=mask, inplace=True )
df.head(5)

## Calcul de variables liées à la surface et au périmètre
Les pertes de chaleur sont liées à la surface d'échange avec l'extérieur, qui ici correspond principalement aux murs et donc au périmètre de chaque bâtiment multiplié par la hauteur non connue ici, mais probablement standardisée.

Deux variables sont donc calculées : le GFA per floor et sa racine carrée, qui estime le périmètre pour un carré (inexacte mais possiblement mieux que rien).

In [ ]:
df = df.copy() # suppress warning
df['GFA_per_floor'] = GFA_ref / ( df['NumberofBuildings'].values * df['NumberofFloors'].values )
df['sqrt_GFA_per_floor'] = np.sqrt( df['GFA_per_floor'].values )
df.head(5)

Les variables targets sont mises à la fin du DataFrame.

In [ ]:
targets = ['SiteEnergyUseWN(kBtu)', 'TotalGHGEmissionsWN']
print( targets, 'in DataFrame:', [key in df.columns for key in targets])

# les colonnes targets sont mises à la fin
df = df[ [key for key in df.columns if not key in targets ] + targets ]
df.head(5)

## Corrélation des nouvelles variables

In [ ]:
tmp = pd.DataFrame()
tmp['GFA Building(s)'] = GFA_ref
tmp2 = df.select_dtypes(include=[np.number]).copy()
tmp[tmp2.columns] = tmp2.values 
tmp.iloc[:,-2:] = tmp.iloc[:,-2:].values / GFA_ref.reshape(-1,1)



tmp.rename( columns={var:f'{var}_per_GFA' for var in tmp.columns.tolist()[-2:] }, inplace=True )
display( tmp.columns )
corr = tmp.corr()

mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True
fig, ax = plt.subplots(figsize=(21*cm,18*cm))
ax = sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={'size':6}, 
                 mask=mask, center=0, cmap="coolwarm")
plt.title(f"Heatmap des corrélations linéaires\n")
plt.show()

In [ ]:
variables = corr.columns.values[ ~corr.columns.str.contains( 'SiteEnergy|Emissi') ]
variables

In [ ]:
tmp2 = tmp[variables]

tmp_vif = pd.DataFrame()
tmp_vif["feature"] = tmp2.columns
tmp_vif["VIF"] = [variance_inflation_factor(tmp2.values, i) 
                   for i in range(len(tmp2.columns))]

display(tmp_vif[tmp_vif['VIF'] > 5])

In [ ]:
mask = 'PropertyGFABuilding(s)'
print( f'{mask} in df_annexe:', mask in df_annexe.columns )

## Scatter plots

In [ ]:
y,y2 = tmp.iloc[:,-2:].values.T

# outlier sur les cibles
loc = y > 1100
sr_loc
display( df_annexe.loc[loc, 'PropertyName'])
display( df.loc[ loc, : ] )

for var in variables:
    fig, axs = plt.subplots( ncols=2, figsize=(20*cm,8*cm) )
    x = tmp[var].values
    axs[0].plot( x, y, 'bo')
    axs[1].plot( x, y2, 'bo')
    axs[0].set_xlabel(var)
    axs[1].set_xlabel(var)
    axs[0].set_ylabel(targets[0] + ' per GFA')
    axs[1].set_ylabel(targets[1] + ' per GFA')
    axs[0].set_title( 'corrélation: {:.1f}%'.format( 100* st.pearsonr(x,y)[0] ) )
    axs[1].set_title( 'corrélation: {:.1f}%'.format( 100* st.pearsonr(x,y)[1] ) )
    fig.tight_layout()

## histogramme / transformation

In [ ]:
exponents = [1, 0.01, 0.5, 0.5, 0.5, 0.5, 0.5, 0.1, 0.1, 0.1]

display( df[variables[1:]].describe() )
# for var, expo in zip(variables[1:], exponents):
for var in variables[1:]:
    values = df[var]
    n_zeros = (values==0).sum()
    values = values[values !=0]
    if '%' in var:
        n_ones = (values==1.).sum()
        values = values[values!=1]
    kurtosis = st.kurtosis( values )
    skew = st.skew( values )

    suptitle = f'{var}: {n_zeros} zeros'
    if '%' in var:
        suptitle += f', {n_ones} ones'

    fig, axs = plt.subplots( ncols=2, figsize=(18*cm,8*cm))
    fig.suptitle( suptitle )
    axs[0].hist( values, bins=50 )
    axs[0].set_xlabel( 'values'  )
    axs[0].set_ylabel( 'count' )
    axs[0].set_title( f'skweness: {skew:.3f}, kurtosis: {kurtosis:.3f}')

    values_log = np.log(values + (1 - min(0,values.min())))
    kurtosis = st.kurtosis( values_log )
    skew = st.skew( values_log )
    axs[1].hist( values_log, bins=50 )
    axs[1].set_xlabel( 'log(values)' )
    axs[1].set_ylabel( 'count' )
    axs[1].set_title( f'skweness: {skew:.3f}, kurtosis: {kurtosis:.3f}')

    # values_pow = values**expo
    # kurtosis = st.kurtosis( values_pow )
    # skew = st.skew( values_pow )
    # axs[2].hist( values_pow, bins=50 )
    # axs[2].set_xlabel( f'values**{expo}' )
    # axs[2].set_ylabel( 'count' )
    # axs[2].set_title( f'skweness: {skew:.3f}, kurtosis: {kurtosis:.3f}')

    fig.tight_layout()


In [ ]:
features = [ feature for feature in df.columns if not feature in vars_to_delete ]

print( features )

features_log = df.columns[df.columns.str.contains( '|'.join(['per_floor','Numberof']) )].tolist()
print('\nfeatures à passer au log:\n', features_log )

## 5.1 OneHotEncoding

In [ ]:
value_counts = df['Type_use_1'].value_counts().add(
    df['Type_use_2'].value_counts(), fill_value=0 ).add(
    df['Type_use_3'].value_counts(), fill_value=0 ).astype(int)

PropertyUseTypes = value_counts.index
PropertyUseTypes = PropertyUseTypes.drop( 'parking' ) # valeur redondante avec % GFA Parking ?
print('propery use types:', PropertyUseTypes.tolist() )

X_propotion_PropertyUseTypeGFA = np.zeros( (len(df), len(PropertyUseTypes)) )
# X_ParkingGFA = np.zeros( len(df) )
for i, index in enumerate(df.index):
    for j in range(1,4) :
        usetype = df.at[index, f'Type_use_{j}']
        if isinstance( usetype, float ): # test if isnan
            break
        if usetype == 'parking':
            # X_ParkingGFA[i] += df.at[ index, 'LargestPropertyUseTypeGFA']
            continue
        k = PropertyUseTypes.get_loc( usetype )
        X_propotion_PropertyUseTypeGFA[i,k] += df.at[index, f'prop_use_{j}']

PropertyUseTypes = [ f'ohe0 {var}' for var in PropertyUseTypes]

In [ ]:
df_hotencoding_usetype = pd.DataFrame( X_propotion_PropertyUseTypeGFA, columns=PropertyUseTypes, index=df.index )
print( 'DataFrame usetype shape:', df_hotencoding_usetype.shape )
df_hotencoding_usetype.head(10).round(2)

In [ ]:
key = 'Neighborhood'
df[key] = df[key].astype('category')
print( df[key].cat.categories )
features_location = [ f'ohe1 {var}' for var in df[key].cat.categories.tolist()]

X_location = np.zeros( (len(df), len(df[key].cat.categories) ) )
X_location[range(len(df)), df[key].cat.codes] = 1.
df_hotencoding_location = pd.DataFrame( X_location, columns=features_location, index=df.index )

print( 'DataFrame location shape:', df_hotencoding_location.shape )
display(df_hotencoding_location.sample(5) )

In [ ]:
print('DataFrame shape:', df.shape )
df.drop( columns= ['Neighborhood', 'Type_use_1', 'prop_use_1', 
    'Type_use_2', 'prop_use_2', 'Type_use_3', 'prop_use_3'], inplace=True )
print('DataFrame shape:', df.shape )

In [ ]:
df.columns

# 6. Sauvegarde

In [ ]:
path = 'data/cleaned/'
filename = '2016_Building_Energy_Benchmarking'
compression = 'gzip'

with open( path + filename + 'features_log', 'wb') as file:
    pickle.dump( features_log , file )

df.to_pickle( r'{:}{:}.pkl'.format(path, filename), compression=compression)
df_annexe.to_pickle( r'{:}{:}_annexe.pkl'.format(path, filename), compression=compression)
df_hotencoding_usetype.to_pickle( r'{:}{:}_usetype.pkl'.format(path, filename), compression=compression)
df_hotencoding_location.to_pickle( r'{:}{:}_location.pkl'.format(path, filename), compression=compression)